## PII Masking Baseline with BiLSTM

This notebook replaces the old binary private/non-private classifier with a token-level PII masking baseline. The goal is to keep useful safe text while masking only private spans before anything is passed forward.

### 1. Imports, paths, and hyperparameters

This notebook is configured for a real baseline run by default. It uses full examples, dynamic padding during batching, type-specific BIO labels, and saves artifacts to `model/artifacts/`.

In [ ]:
import ast
import json
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset

SEED = 42
DATASET_NAME = "ai4privacy/pii-masking-300k"
LANGUAGE_FILTER = "en"  # Set to None to keep all languages.
TRAIN_FRACTION = 0.8
BATCH_SIZE = 32
EMBEDDING_DIM = 64
HIDDEN_DIM = 64
DROPOUT = 0.2
EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "model").exists():
            return candidate
    raise RuntimeError("Could not find repo root. Run this notebook from the repo root or model/baseline_model.")

REPO_ROOT = find_repo_root()
ARTIFACT_DIR = REPO_ROOT / "model" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Repo root:", REPO_ROOT)
print("Artifact directory:", ARTIFACT_DIR)
print("Device:", device)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)

### 2. Load the dataset and inspect columns

The model input is `source_text`. The masked `target_text` is not used as input because that would leak the answer into training. It can be used only as a reference output.

In [ ]:
dataset = load_dataset(DATASET_NAME)

print(dataset)
for split_name, split_data in dataset.items():
    print(f"{split_name}: {split_data.num_rows:,} rows")
    print("Columns:", split_data.column_names)

main_split_name = "train" if "train" in dataset else list(dataset.keys())[0]
raw_df = dataset[main_split_name].to_pandas()

print("\nUsing split:", main_split_name)
print("Raw rows:", len(raw_df))
print("Raw columns:", list(raw_df.columns))

sample_summary = {column: type(raw_df.iloc[0][column]).__name__ for column in raw_df.columns}
print("\nSample value types:")
print(sample_summary)

### 3. Clean and filter rows

The cleaning is intentionally simple: strip whitespace, normalize repeated whitespace, remove rows with missing or empty `source_text`, and remove duplicate `source_text` values. Short but valid examples are kept. If a language column exists, `LANGUAGE_FILTER` controls whether we keep only English or keep all languages.

In [ ]:
def normalize_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def detect_language_column(df):
    for column in ["language", "lang", "locale"]:
        if column in df.columns:
            return column
    return None

if "source_text" not in raw_df.columns:
    raise ValueError("Expected a source_text column in the dataset.")

df = raw_df.copy()
df = df[df["source_text"].notna()].copy()
df["source_text"] = df["source_text"].apply(normalize_text)
df = df[df["source_text"].str.len() > 0].copy()

language_column = detect_language_column(df)
if LANGUAGE_FILTER and language_column:
    before = len(df)
    allowed = {LANGUAGE_FILTER.lower(), "english" if LANGUAGE_FILTER.lower() == "en" else LANGUAGE_FILTER.lower()}
    df = df[df[language_column].astype(str).str.lower().isin(allowed)]
    print(f"Language filter kept {len(df):,} of {before:,} rows using {language_column}={LANGUAGE_FILTER!r}.")
elif LANGUAGE_FILTER:
    print("No language column found; keeping all rows.")

df = df.drop_duplicates(subset=["source_text"]).reset_index(drop=True)

print("Rows after cleaning/filtering:", len(df))
print("Columns available for labels:", [c for c in ["privacy_mask", "span_labels", "mberttokens", "mbert_bio_labels", "target_text"] if c in df.columns])
length_preview = df["source_text"].str.len().head(5).tolist()
print("Example source_text lengths:", length_preview)

### 4. Convert privacy spans into token labels

The preferred path is `source_text` plus `privacy_mask`, with `span_labels` as a fallback. The tokenizer keeps character offsets so privacy spans become type-specific BIO labels such as `B-EMAIL`, `I-NAME`, or `B-PHONE_NUMBER`. `O` means the token is not private.

In [ ]:
TOKEN_PATTERN = re.compile(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}|\d+[\d.,:/+-]*|[A-Za-z]+(?:[-'][A-Za-z]+)*|\w+|[^\w\s]", re.UNICODE)

LABEL_ALIASES = {
    "FIRSTNAME": "NAME",
    "LASTNAME": "NAME",
    "GIVENNAME": "NAME",
    "SURNAME": "NAME",
    "EMAIL_ADDRESS": "EMAIL",
    "PHONE": "PHONE_NUMBER",
    "PHONENUMBER": "PHONE_NUMBER",
    "MOBILEPHONE": "PHONE_NUMBER",
    "CITY_NAME": "CITY",
    "LOCATION_ADDRESS": "LOCATION",
    "STREET_ADDRESS": "ADDRESS",
    "ID_CARD": "IDCARD",
    "NATIONAL_ID": "IDCARD",
}

def parse_annotation_value(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except (SyntaxError, ValueError):
            return []
    return []

def normalize_entity_label(label):
    label = str(label or "PRIVATE").strip()
    label = re.sub(r"^[BI]-", "", label, flags=re.IGNORECASE)
    label = re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_").upper()
    label = LABEL_ALIASES.get(label, label)
    return label or "PRIVATE"

def get_dict_value(item, keys):
    for key in keys:
        if key in item and item[key] not in [None, ""]:
            return item[key]
    return None

def find_value_span(text, value, start_at=0):
    if not value:
        return None
    value = str(value)
    start = text.find(value, start_at)
    if start == -1:
        start = text.find(value)
    if start == -1:
        return None
    return start, start + len(value)

def extract_spans_from_privacy_mask(text, value):
    spans = []
    cursor = 0
    for item in parse_annotation_value(value):
        if not isinstance(item, dict):
            continue
        label = normalize_entity_label(get_dict_value(item, ["label", "entity", "entity_type", "type"]))
        start = get_dict_value(item, ["start", "start_offset", "begin"])
        end = get_dict_value(item, ["end", "end_offset", "stop"])
        mask_value = get_dict_value(item, ["value", "text", "source_value", "original"])

        if start is None or end is None:
            found = find_value_span(text, mask_value, cursor)
            if not found:
                continue
            start, end = found
        else:
            start, end = int(start), int(end)

        if 0 <= start < end <= len(text):
            spans.append({"start": start, "end": end, "label": label})
            cursor = end
    return spans

def extract_spans_from_span_labels(text, value):
    spans = []
    for item in parse_annotation_value(value):
        label = None
        start = None
        end = None
        if isinstance(item, dict):
            label = get_dict_value(item, ["label", "entity", "entity_type", "type"])
            start = get_dict_value(item, ["start", "start_offset", "begin"])
            end = get_dict_value(item, ["end", "end_offset", "stop"])
        elif isinstance(item, (list, tuple)) and len(item) >= 3:
            if isinstance(item[0], (int, np.integer)):
                start, end, label = item[0], item[1], item[2]
            else:
                label, start, end = item[0], item[1], item[2]
        if start is None or end is None:
            continue
        start, end = int(start), int(end)
        if 0 <= start < end <= len(text):
            spans.append({"start": start, "end": end, "label": normalize_entity_label(label)})
    return spans

def tokenize_with_offsets(text):
    tokens = []
    offsets = []
    for match in TOKEN_PATTERN.finditer(text):
        tokens.append(match.group(0))
        offsets.append((match.start(), match.end()))
    return tokens, offsets

def spans_to_bio_labels(offsets, spans):
    labels = ["O"] * len(offsets)
    for span in sorted(spans, key=lambda item: (item["start"], item["end"])):
        first_token = True
        entity = span["label"]
        for i, (token_start, token_end) in enumerate(offsets):
            overlaps = token_start < span["end"] and token_end > span["start"]
            if overlaps:
                prefix = "B" if first_token else "I"
                labels[i] = f"{prefix}-{entity}"
                first_token = False
    return labels

def build_labeled_example(row):
    text = row["source_text"]
    spans = []
    if "privacy_mask" in row.index:
        spans = extract_spans_from_privacy_mask(text, row["privacy_mask"])
    if not spans and "span_labels" in row.index:
        spans = extract_spans_from_span_labels(text, row["span_labels"])
    tokens, offsets = tokenize_with_offsets(text)
    labels = spans_to_bio_labels(offsets, spans)
    return {
        "source_text": text,
        "tokens": tokens,
        "offsets": offsets,
        "labels": labels,
        "spans": spans,
        "target_text": row["target_text"] if "target_text" in row.index else None,
    }

examples = [build_labeled_example(row) for _, row in df.iterrows()]
examples = [ex for ex in examples if ex["tokens"] and ex["spans"] and len(ex["tokens"]) == len(ex["labels"])]

entity_counter = Counter(span["label"] for ex in examples for span in ex["spans"])
token_label_counter = Counter(label for ex in examples for label in ex["labels"])

print("Labeled examples:", len(examples))
print("Top original entity labels:", entity_counter.most_common(15))
print("Top token labels:", token_label_counter.most_common(15))
if examples:
    first = examples[0]
    print("Example token count:", len(first["tokens"]))
    print("Example label count:", len(first["labels"]))
    print("Example span count:", len(first["spans"]))

### 5. Build mappings and train/validation split

The model sees token IDs and predicts one BIO label per token. Obvious patterns such as email addresses and long numbers are normalized before vocabulary lookup so raw-text inference can generalize beyond exact training tokens. `O` means the token is not PII. Any `B-*` or `I-*` label is treated as a private token for recall/F1.

In [ ]:
if not examples:
    raise ValueError("No usable labeled examples were created. Inspect privacy_mask/span_labels parsing above.")

EMAIL_PATTERN = re.compile(r"^[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}$")
NUMBER_PATTERN = re.compile(r"^\d+[\d.,:/+-]*$")
ALNUM_PATTERN = re.compile(r"^(?=.*[A-Za-z])(?=.*\d)[A-Za-z0-9_.+-]+$")

def token_feature(token):
    if EMAIL_PATTERN.match(token):
        return "<EMAIL_PATTERN>"
    if NUMBER_PATTERN.match(token):
        return "<NUMBER_PATTERN>"
    if ALNUM_PATTERN.match(token):
        return "<ALNUM_PATTERN>"
    return token.lower()

random.shuffle(examples)
split_index = max(1, int(len(examples) * TRAIN_FRACTION))
train_examples = examples[:split_index]
val_examples = examples[split_index:] or examples[: min(100, len(examples))]

token_counter = Counter(token_feature(token) for ex in train_examples for token in ex["tokens"])
token_to_id = {"<PAD>": 0, "<UNK>": 1}
for token, count in token_counter.most_common():
    token_to_id[token] = len(token_to_id)

all_labels = sorted({label for ex in examples for label in ex["labels"]})
if "O" in all_labels:
    all_labels.remove("O")
all_labels = ["O", *all_labels]
label_to_id = {label: i for i, label in enumerate(all_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("Vocabulary size:", len(token_to_id))
print("Number of labels:", len(label_to_id))
print("Labels:", list(label_to_id.keys())[:30])

### 6. Dataset and DataLoader

Each full example is kept intact. The DataLoader uses dynamic padding to pad only to the longest sequence in each batch. Padding labels use `-100` so PyTorch ignores them in the loss.

In [ ]:
def encode_tokens(tokens):
    return [token_to_id.get(token_feature(token), token_to_id["<UNK>"]) for token in tokens]

def encode_labels(labels):
    return [label_to_id[label] for label in labels]

class PIITokenDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        return {
            "input_ids": torch.tensor(encode_tokens(example["tokens"]), dtype=torch.long),
            "labels": torch.tensor(encode_labels(example["labels"]), dtype=torch.long),
        }

def collate_batch(batch):
    max_length = max(item["input_ids"].size(0) for item in batch)
    input_ids = torch.zeros((len(batch), max_length), dtype=torch.long)
    labels = torch.full((len(batch), max_length), fill_value=-100, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_length), dtype=torch.bool)

    for i, item in enumerate(batch):
        length = item["input_ids"].size(0)
        input_ids[i, :length] = item["input_ids"]
        labels[i, :length] = item["labels"]
        attention_mask[i, :length] = True

    return input_ids, labels, attention_mask

train_loader = DataLoader(PIITokenDataset(train_examples), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(PIITokenDataset(val_examples), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

batch_input_ids, batch_labels, batch_mask = next(iter(train_loader))
print("Batch input shape:", tuple(batch_input_ids.shape))
print("Batch label shape:", tuple(batch_labels.shape))

### 7. BiLSTM token tagger

The model is intentionally small: token IDs go through an embedding layer, a bidirectional LSTM, and a linear classifier that predicts one label per token.

In [ ]:
class BiLSTMTokenTagger(nn.Module):
    def __init__(self, vocab_size, num_labels, embedding_dim, hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        output, _ = self.lstm(embedded)
        output = self.dropout(output)
        return self.classifier(output)

model = BiLSTMTokenTagger(
    vocab_size=len(token_to_id),
    num_labels=len(label_to_id),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
).to(device)

class_weights = torch.ones(len(label_to_id), dtype=torch.float32)
class_weights[label_to_id["O"]] = 0.25
criterion = nn.CrossEntropyLoss(ignore_index=-100, weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

### 8. Metrics

Token accuracy is useful, but PII recall matters more because missed private tokens can leak information. Precision, recall, and F1 below treat any non-`O` label as a PII token.

In [ ]:
def calculate_token_metrics(true_ids, pred_ids):
    true_ids = np.array(true_ids)
    pred_ids = np.array(pred_ids)
    if len(true_ids) == 0:
        return {"accuracy": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0, "tp": 0, "fp": 0, "fn": 0}

    o_id = label_to_id["O"]
    true_pii = true_ids != o_id
    pred_pii = pred_ids != o_id

    tp = int((true_pii & pred_pii).sum())
    fp = int((~true_pii & pred_pii).sum())
    fn = int((true_pii & ~pred_pii).sum())
    accuracy = float((true_ids == pred_ids).mean())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

def run_epoch(dataloader, training):
    model.train(training)
    total_loss = 0.0
    total_tokens = 0
    all_true = []
    all_pred = []

    for input_ids, labels, attention_mask in dataloader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        attention_mask = attention_mask.to(device)

        if training:
            optimizer.zero_grad()

        logits = model(input_ids)
        loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))

        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        valid_positions = attention_mask & (labels != -100)
        predictions = logits.argmax(dim=-1)
        true_batch = labels[valid_positions].detach().cpu().numpy().tolist()
        pred_batch = predictions[valid_positions].detach().cpu().numpy().tolist()

        all_true.extend(true_batch)
        all_pred.extend(pred_batch)
        total_loss += loss.item() * len(true_batch)
        total_tokens += len(true_batch)

    metrics = calculate_token_metrics(all_true, all_pred)
    metrics["loss"] = total_loss / total_tokens if total_tokens else 0.0
    return metrics

### 9. Train and evaluate the baseline

This cell trains the BiLSTM baseline for the configured number of epochs and evaluates on the validation split after each epoch. For PII masking, F1 and especially recall are more important than raw accuracy because most tokens are usually non-private.

In [ ]:
history = []
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, training=True)
    with torch.no_grad():
        val_metrics = run_epoch(val_loader, training=False)

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
    }
    history.append(row)
    print(row)

history_df = pd.DataFrame(history)
display(history_df)

final_eval_metrics = history[-1]
print("Final validation metrics")
print(f"Validation loss: {final_eval_metrics['val_loss']:.4f}")
print(f"Token accuracy:  {final_eval_metrics['val_accuracy']:.4f}")
print(f"Precision:       {final_eval_metrics['val_precision']:.4f}")
print(f"Recall:          {final_eval_metrics['val_recall']:.4f}")
print(f"F1-score:        {final_eval_metrics['val_f1']:.4f}")

### 10. Prediction helpers

`mask_text(...)` accepts normal raw text, tokenizes it with the same tokenizer, predicts token labels, merges adjacent predicted PII tokens, and replaces each span with an entity mask.

In [ ]:
def label_to_entity(label):
    if label == "O":
        return None
    return normalize_entity_label(label.split("-", 1)[-1])

def predict_labels_for_text(text):
    text = normalize_text(text)
    tokens, offsets = tokenize_with_offsets(text)
    if not tokens:
        return text, tokens, offsets, []

    input_ids = torch.tensor([encode_tokens(tokens)], dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(input_ids)
        pred_ids = logits.argmax(dim=-1).squeeze(0).cpu().numpy().tolist()
    pred_labels = [id_to_label[pred_id] for pred_id in pred_ids]
    return text, tokens, offsets, pred_labels

def prediction_spans(offsets, labels):
    spans = []
    current = None
    for (start, end), label in zip(offsets, labels):
        entity = label_to_entity(label)
        if entity is None:
            if current:
                spans.append(current)
                current = None
            continue

        begins_new = label.startswith("B-") or current is None or current["label"] != entity or start > current["end"] + 1
        if begins_new:
            if current:
                spans.append(current)
            current = {"start": start, "end": end, "label": entity}
        else:
            current["end"] = end

    if current:
        spans.append(current)
    return spans

def apply_masks(text, spans):
    if not spans:
        return text
    pieces = []
    cursor = 0
    for span in spans:
        pieces.append(text[cursor:span["start"]])
        pieces.append(f"[{span['label']}]")
        cursor = span["end"]
    pieces.append(text[cursor:])
    return normalize_text("".join(pieces))

def mask_text(text, return_details=False):
    normalized, tokens, offsets, pred_labels = predict_labels_for_text(text)
    spans = prediction_spans(offsets, pred_labels)
    masked = apply_masks(normalized, spans)
    if return_details:
        return {
            "input_text": normalized,
            "masked_text": masked,
            "tokens": tokens,
            "predicted_labels": pred_labels,
            "predicted_spans": spans,
        }
    return masked

### 11. Qualitative masking examples

These examples come from the validation split. They show the full source text, the dataset target/mask, and the model-predicted masked text so the masking behavior can be checked visually.

In [ ]:
def has_text_value(value):
    if value is None:
        return False
    if isinstance(value, float) and pd.isna(value):
        return False
    return bool(str(value).strip())

def reference_mask_for_example(example):
    target_text = example.get("target_text")
    if has_text_value(target_text):
        return normalize_text(target_text)
    return apply_masks(example["source_text"], example["spans"])

qualitative_rows = []
for example in val_examples[:5]:
    qualitative_rows.append(
        {
            "original_source_text": example["source_text"],
            "dataset_target_or_mask": reference_mask_for_example(example),
            "model_predicted_mask": mask_text(example["source_text"]),
        }
    )

with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(qualitative_rows))

### 12. Save trained model

The final block saves the trained model weights, model configuration, preprocessing configuration, token vocabulary, label map, and final validation metrics.

In [ ]:
model_path = ARTIFACT_DIR / "pii_masking_bilstm.pt"
model_config_path = ARTIFACT_DIR / "pii_masking_bilstm_config.json"
vocab_path = ARTIFACT_DIR / "token_vocab.json"
label_map_path = ARTIFACT_DIR / "label_map.json"
config_path = ARTIFACT_DIR / "preprocessing_config.json"
metrics_path = ARTIFACT_DIR / "final_eval_metrics.json"

model_config = {
    "architecture": "Embedding -> BiLSTM -> Linear token classifier",
    "vocab_size": len(token_to_id),
    "num_labels": len(label_to_id),
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "train_fraction": TRAIN_FRACTION,
}

preprocessing_config = {
    "dataset_name": DATASET_NAME,
    "source_column": "source_text",
    "label_sources": [column for column in ["privacy_mask", "span_labels"] if column in df.columns],
    "target_text_usage": "reference_only_not_training_input" if "target_text" in df.columns else "not_available",
    "language_filter": LANGUAGE_FILTER,
    "text_truncation": "disabled",
    "padding": "dynamic per batch in collate_batch",
    "token_pattern": TOKEN_PATTERN.pattern,
    "token_feature_rules": ["EMAIL_PATTERN", "NUMBER_PATTERN", "ALNUM_PATTERN", "lowercase fallback"],
    "label_format": "type-specific BIO labels from privacy_mask/span_labels",
}

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_config": model_config,
        "preprocessing_config": preprocessing_config,
        "label_to_id": label_to_id,
        "token_to_id": token_to_id,
        "final_eval_metrics": final_eval_metrics,
    },
    model_path,
)

with open(model_config_path, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=2)

with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(token_to_id, f, indent=2)

id_to_label_json = {str(label_id): label for label_id, label in id_to_label.items()}
with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump({"label_to_id": label_to_id, "id_to_label": id_to_label_json}, f, indent=2)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(preprocessing_config, f, indent=2)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(final_eval_metrics, f, indent=2)

print("Saved model:", model_path)
print("Saved model config:", model_config_path)
print("Saved vocabulary:", vocab_path)
print("Saved label map:", label_map_path)
print("Saved preprocessing config:", config_path)
print("Saved final evaluation metrics:", metrics_path)